# Math Ops Library: Python usage examples (ctypes, strict mode)

This notebook demonstrates all 20 functions from the C library `math_ops`.
Strict mode: no Python fallback is used. A compiled shared library is required.

In [1]:
from pathlib import Path
import ctypes
import os
import sys

def find_math_ops_library() -> Path:
    ext = '.dll' if sys.platform.startswith('win') else ('.dylib' if sys.platform == 'darwin' else '.so')
    candidates = []
    env_path = os.getenv('MATH_OPS_LIB')
    if env_path:
        candidates.append(Path(env_path))
    cwd = Path.cwd()
    candidates.extend([
        cwd / f'math_ops{ext}',
        cwd / 'build' / f'math_ops{ext}',
        cwd / 'build' / 'Debug' / f'math_ops{ext}',
        cwd / 'build' / 'Release' / f'math_ops{ext}',
    ])
    for p in candidates:
        if p.exists():
            return p
    names = ', '.join(str(p) for p in candidates)
    raise FileNotFoundError('Strict mode error: math_ops shared library was not found. Checked: ' + names)

def as_rows(flat, rows, cols):
    out = []
    for i in range(rows):
        out.append([float(flat[i * cols + j]) for j in range(cols)])
    return out

lib_path = find_math_ops_library()
lib = ctypes.CDLL(str(lib_path))
print(f'Loaded native library: {lib_path}')

Loaded native library: C:\projects\низкоуровневая прога\math_ops.dll


In [2]:
MO_OK = 0
MO_ERR_NULL_PTR = -1
MO_ERR_INVALID_ARG = -2
MO_ERR_OVERFLOW = -3
MO_ERR_ALLOC = -4

c_double = ctypes.c_double
c_int = ctypes.c_int
c_uint = ctypes.c_uint
c_ull = ctypes.c_ulonglong
c_ll = ctypes.c_longlong
c_size_t = ctypes.c_size_t
c_double_p = ctypes.POINTER(c_double)
c_double_pp = ctypes.POINTER(c_double_p)

lib.mo_add.argtypes = [c_double, c_double]
lib.mo_add.restype = c_double
lib.mo_subtract.argtypes = [c_double, c_double]
lib.mo_subtract.restype = c_double
lib.mo_multiply.argtypes = [c_double, c_double]
lib.mo_multiply.restype = c_double
lib.mo_divide.argtypes = [c_double, c_double, c_double_p]
lib.mo_divide.restype = c_int
lib.mo_power.argtypes = [c_double, c_int]
lib.mo_power.restype = c_double
lib.mo_abs.argtypes = [c_double]
lib.mo_abs.restype = c_double
lib.mo_sqrt.argtypes = [c_double, c_double_p]
lib.mo_sqrt.restype = c_int
lib.mo_factorial.argtypes = [c_uint, ctypes.POINTER(c_ull)]
lib.mo_factorial.restype = c_int
lib.mo_gcd.argtypes = [c_int, c_int, ctypes.POINTER(c_int)]
lib.mo_gcd.restype = c_int
lib.mo_lcm.argtypes = [c_int, c_int, ctypes.POINTER(c_ll)]
lib.mo_lcm.restype = c_int
lib.mo_matrix_multiply_2x2.argtypes = [c_double_p, c_double_p, c_double_p]
lib.mo_matrix_multiply_2x2.restype = c_int
lib.mo_matrix_multiply_3x3.argtypes = [c_double_p, c_double_p, c_double_p]
lib.mo_matrix_multiply_3x3.restype = c_int
lib.mo_matrix_determinant_3x3.argtypes = [c_double_p, c_double_p]
lib.mo_matrix_determinant_3x3.restype = c_int
lib.mo_matrix_inverse_2x2.argtypes = [c_double_p, c_double_p]
lib.mo_matrix_inverse_2x2.restype = c_int
lib.mo_solve_linear_2x2.argtypes = [c_double_p, c_double_p, c_double_p]
lib.mo_solve_linear_2x2.restype = c_int
lib.mo_matrix_alloc.argtypes = [c_size_t, c_size_t, c_double_pp]
lib.mo_matrix_alloc.restype = c_int
lib.mo_matrix_free.argtypes = [c_double_p]
lib.mo_matrix_free.restype = c_int
lib.mo_matrix_fill.argtypes = [c_double_p, c_size_t, c_size_t, c_double]
lib.mo_matrix_fill.restype = c_int
lib.mo_matrix_copy.argtypes = [c_double_p, c_size_t, c_size_t, c_double_pp]
lib.mo_matrix_copy.restype = c_int
lib.mo_matrix_multiply_dyn.argtypes = [c_double_p, c_double_p, c_size_t, c_size_t, c_size_t, c_double_pp]
lib.mo_matrix_multiply_dyn.restype = c_int
print('Backend: native (strict)')

Backend: native (strict)


In [3]:
# 1) mo_add
print('mo_add(10.5, 2.5) =', lib.mo_add(10.5, 2.5))

mo_add(10.5, 2.5) = 13.0


In [4]:
# 2) mo_subtract
print('mo_subtract(10.5, 2.5) =', lib.mo_subtract(10.5, 2.5))

mo_subtract(10.5, 2.5) = 8.0


In [5]:
# 3) mo_multiply
print('mo_multiply(6.0, 7.0) =', lib.mo_multiply(6.0, 7.0))

mo_multiply(6.0, 7.0) = 42.0


In [6]:
# 4) mo_divide
div_result = c_double()
status = lib.mo_divide(22.0, 7.0, ctypes.byref(div_result))
print('status =', status, '| result =', div_result.value)

status = 0 | result = 3.142857142857143


In [7]:
# 5) mo_power
print('mo_power(2.0, 10) =', lib.mo_power(2.0, 10))

mo_power(2.0, 10) = 1024.0


In [8]:
# 6) mo_abs
print('mo_abs(-123.456) =', lib.mo_abs(-123.456))

mo_abs(-123.456) = 123.456


In [9]:
# 7) mo_sqrt
sqrt_result = c_double()
status = lib.mo_sqrt(144.0, ctypes.byref(sqrt_result))
print('status =', status, '| result =', sqrt_result.value)

status = 0 | result = 12.0


In [10]:
# 8) mo_factorial
factorial_result = c_ull()
status = lib.mo_factorial(10, ctypes.byref(factorial_result))
print('status =', status, '| result =', factorial_result.value)

status = 0 | result = 3628800


In [11]:
# 9) mo_gcd
gcd_result = c_int()
status = lib.mo_gcd(84, 30, ctypes.byref(gcd_result))
print('status =', status, '| result =', gcd_result.value)

status = 0 | result = 6


In [12]:
# 10) mo_lcm
lcm_result = c_ll()
status = lib.mo_lcm(21, 6, ctypes.byref(lcm_result))
print('status =', status, '| result =', lcm_result.value)

status = 0 | result = 42


In [13]:
# 11) mo_matrix_multiply_2x2
a2 = (c_double * 4)(1.0, 2.0, 3.0, 4.0)
b2 = (c_double * 4)(2.0, 0.0, 1.0, 2.0)
r2 = (c_double * 4)()
status = lib.mo_matrix_multiply_2x2(a2, b2, r2)
print('status =', status, '| result =', as_rows(r2, 2, 2))

status = 0 | result = [[4.0, 4.0], [10.0, 8.0]]


In [14]:
# 12) mo_matrix_multiply_3x3
a3 = (c_double * 9)(1.0, 2.0, 3.0, 0.0, 1.0, 4.0, 5.0, 6.0, 0.0)
b3 = (c_double * 9)(7.0, 8.0, 9.0, 2.0, 3.0, 4.0, 1.0, 0.0, 6.0)
r3 = (c_double * 9)()
status = lib.mo_matrix_multiply_3x3(a3, b3, r3)
print('status =', status, '| result =', as_rows(r3, 3, 3))

status = 0 | result = [[14.0, 14.0, 35.0], [6.0, 3.0, 28.0], [47.0, 58.0, 69.0]]


In [15]:
# 13) mo_matrix_determinant_3x3
m3 = (c_double * 9)(1.0, 2.0, 3.0, 0.0, 1.0, 4.0, 5.0, 6.0, 0.0)
det3 = c_double()
status = lib.mo_matrix_determinant_3x3(m3, ctypes.byref(det3))
print('status =', status, '| determinant =', det3.value)

status = 0 | determinant = 1.0


In [16]:
# 14) mo_matrix_inverse_2x2
inv_in = (c_double * 4)(4.0, 7.0, 2.0, 6.0)
inv_out = (c_double * 4)()
status = lib.mo_matrix_inverse_2x2(inv_in, inv_out)
print('status =', status, '| inverse =', as_rows(inv_out, 2, 2))

status = 0 | inverse = [[0.6000000000000001, -0.7000000000000001], [-0.2, 0.4]]


In [17]:
# 15) mo_solve_linear_2x2
A = (c_double * 4)(2.0, 1.0, 5.0, 3.0)
b = (c_double * 2)(1.0, 2.0)
x = (c_double * 2)()
status = lib.mo_solve_linear_2x2(A, b, x)
print('status =', status, '| solution =', [x[0], x[1]])

status = 0 | solution = [1.0, -1.0]


In [18]:
# 16) mo_matrix_alloc
dyn_alloc = c_double_p()
status = lib.mo_matrix_alloc(2, 3, ctypes.byref(dyn_alloc))
print('status =', status, '| allocated =', bool(dyn_alloc))
lib.mo_matrix_free(dyn_alloc)

status = 0 | allocated = True


0

In [19]:
# 17) mo_matrix_fill
dyn_fill = c_double_p()
lib.mo_matrix_alloc(2, 3, ctypes.byref(dyn_fill))
status = lib.mo_matrix_fill(dyn_fill, 2, 3, 2.5)
print('status =', status, '| matrix =', as_rows(dyn_fill, 2, 3))
lib.mo_matrix_free(dyn_fill)

status = 0 | matrix = [[2.5, 2.5, 2.5], [2.5, 2.5, 2.5]]


0

In [20]:
# 18) mo_matrix_copy
src = (c_double * 6)(1.0, 2.0, 3.0, 4.0, 5.0, 6.0)
dyn_copy = c_double_p()
status = lib.mo_matrix_copy(src, 2, 3, ctypes.byref(dyn_copy))
print('status =', status, '| copy =', as_rows(dyn_copy, 2, 3))
lib.mo_matrix_free(dyn_copy)

status = 0 | copy = [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]]


0

In [21]:
# 19) mo_matrix_multiply_dyn
a_dyn = (c_double * 6)(1.0, 2.0, 3.0, 4.0, 5.0, 6.0)
b_dyn = (c_double * 6)(7.0, 8.0, 9.0, 10.0, 11.0, 12.0)
prod_dyn = c_double_p()
status = lib.mo_matrix_multiply_dyn(a_dyn, b_dyn, 2, 3, 2, ctypes.byref(prod_dyn))
print('status =', status, '| product =', as_rows(prod_dyn, 2, 2))
lib.mo_matrix_free(prod_dyn)

status = 0 | product = [[58.0, 64.0], [139.0, 154.0]]


0

In [22]:
# 20) mo_matrix_free
ptr = c_double_p()
lib.mo_matrix_alloc(1, 1, ctypes.byref(ptr))
status = lib.mo_matrix_free(ptr)
print('status =', status)

status = 0
